In [39]:
import pandas as pd
import json

# Carregar dados

erp = pd.read_csv(
    "pedido_erp.csv",
    sep=";",
    quotechar='"'
)

crm = pd.read_csv(
    "clientes_crm.csv",
    sep=";"
)

with open("pedido_ecom.json", "r", encoding="utf-8") as f:
    ecommerce_json = json.load(f)

# transformar json em dataframe
ecommerce = pd.json_normalize(ecommerce_json["docs"])


# remover coluna com lista
ecommerce = ecommerce.drop(columns=["products"])


# renomear colunas
ecommerce = ecommerce.rename(columns={
    "_id": "id_pedido_online",
    "orderNumber": "numero_pedido",
    "customer.doc": "documento",
    "settings.source": "canal",
    "settings.createdAt": "data_pedido",
    "summary.order.parts": "quantidade_itens",
    "summary.order.value": "valor_itens",
    "summary.reserved.parts": "quantidade_reservada",
    "summary.reserved.value": "valor_reservado",
    "summary.reserved.rate": "taxa_reserva",
    "summary.promotionalSubtotal": "subtotal_promocional",
    "summary.additionalValues": "valores_adicionais",
    "summary.discount": "desconto",
    "summary.total": "valor_pedido",
    "summary.subtotal": "subtotal",
    "summary.discountRate": "taxa_desconto",
    "summary.invoiceValue": "valor_fatura",
    "summary.subTotalWithFees": "subtotal_com_taxas",
    "seller.name": "vendedor",
    "summary.coupon.type": "tipo_cupom"
})

erp = erp.rename(columns={
    "id": "id_pedido",
    "number": "numero_pedido",
    "customer_document": "documento",
    "seller_name": "vendedor",
    "order_value": "valor_pedido",
    "order_created": "data_pedido"
})

crm = crm.rename(columns={
    "_id": "id_cliente",
    "document": "documento",
    "name": "nome_cliente",
    "email": "email",
    "ddi": "ddi",
    "ddd": "ddd",
    "phone": "telefone",
    "buy": "permissao_compra",
    "status": "status_cliente",
    "seller_name": "vendedor",
    "created_at": "data_cadastro"
})


# converter tipos
if "data_pedido" in erp.columns:
    erp["data_pedido"] = pd.to_datetime(erp["data_pedido"])

if "valor_pedido" in erp.columns:
    erp["valor_pedido"] = (
        erp["valor_pedido"]
        .astype(str)
        .str.replace(",", ".")
        .astype(float)
    )

if "data_cadastro" in crm.columns:
    crm["data_cadastro"] = pd.to_datetime(crm["data_cadastro"])
    crm["vendedor"] = crm["vendedor"].replace("\\N", "sem vendedor - compra direta")


if "data_pedido" in ecommerce.columns:
    ecommerce["data_pedido"] = pd.to_datetime(ecommerce["data_pedido"])

if "valor_pedido" in ecommerce.columns:
    ecommerce["valor_pedido"] = ecommerce["valor_pedido"].astype(float)
    ecommerce["tipo_cupom"] = ecommerce["tipo_cupom"].fillna("sem cupom")
    ecommerce["documento"] = ecommerce["documento"].fillna("cliente_nao_identificado")
    ecommerce["canal"] = ecommerce["canal"].fillna("não identificado")


erp["vendedor"] = erp["vendedor"].str.strip().str.capitalize()
crm["vendedor"] = crm["vendedor"].str.strip().str.capitalize()
ecommerce["vendedor"] = ecommerce["vendedor"].str.strip().str.capitalize()



# remover duplicatas
erp = erp.drop_duplicates()
crm = crm.drop_duplicates()
ecommerce = ecommerce.drop_duplicates(subset=["id_pedido_online"])


#unir dataframes
erp["origem_venda"] = "ERP"
ecommerce["origem_venda"] = "Ecommerce"

vendas = pd.concat([erp, ecommerce], ignore_index=True)
vendas = vendas.merge(
    crm,
    on="documento",
    how="left")


display(vendas)




,id_pedido,numero_pedido,documento,vendedor_x,valor_pedido,data_pedido,origem_venda,id_pedido_online,canal,quantidade_itens,...,id,nome_cliente,email,ddi,ddd,telefone,permissao_compra,status_cliente,vendedor_y,data_cadastro
0,4140a410-6010-4863-8aec-5c7c15ffd171,2,99.999.999/9999-99,Usuário de teste,1060.80,2016-10-10 18:06:56,ERP,NaN,NaN,NaN,...,780ccd5c-3c5a-40ae-887d-3e1996c1d2ae,Andressa,email@email.com,55.0,\N,911112222.0,not allowed,inactive,Sandra,2016-10-10 18:06:56
1,032ca5d8-7a67-495b-bdeb-f983ba8b0f79,5,99.999.999/9999-99,Usuário de teste,495.50,2016-10-13 13:15:35,ERP,NaN,NaN,NaN,...,780ccd5c-3c5a-40ae-887d-3e1996c1d2ae,Andressa,email@email.com,55.0,\N,911112222.0,not allowed,inactive,Sandra,2016-10-10 18:06:56
2,043a2d5c-5b11-43d1-9d91-97ae933ce5b0,8,68.148.678/0001-50,Usuário de teste,549.00,2016-10-20 13:59:03,ERP,NaN,NaN,NaN,...,08d116e4-58d2-4b4b-9156-ea95a8f34e2c,Guilherme PJ,email@email.com,55.0,\N,911112222.0,allowed,active,Sandra,2016-10-20 13:59:03
3,a46fe3ad-303a-423f-8e3e-4e59d7bc27e4,9,26.153.919/0001-09,Jessica,1328.90,2016-10-27 15:30:19,ERP,NaN,NaN,NaN,...,43bcca1d-266a-4a55-814e-292da4fee161,CAMILA AGUIMAR ARANTES,email@email.com,55.0,\N,911112222.0,allowed,active,Jessica,2016-10-27 15:30:19
4,ffd2fe2c-c49c-4020-8d4c-4c3d49532ba9,10,20.164.518/0001-78,Jessica,1418.90,2016-10-31 15:46:47,ERP,NaN,NaN,NaN,...,4fbd3054-4524-49db-b962-fc2e3d1028cb,Gesiela Fernanda valter,email@email.com,55.0,\N,911112222.0,allowed,active,Jessica,2016-10-31 15:46:47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13079,NaN,18175,04.047.000/0001-06,Natalia,1238.80,2024-11-28 14:31:13+00:00,Ecommerce,673e8a0260c6c3b46e0a4db3,Link,13.0,...,e6b46b88-1f0f-4ee6-bd31-27d18b429fc0,A B MENDES - ME,email@email.com,55.0,91,911112222.0,allowed,active,Natalia,2019-07-16 17:35:51
13080,NaN,18177,52.367.744/0001-42,Natalia r.,3653.67,2024-11-28 23:44:55+00:00,Ecommerce,6748fb85e6fcb94c3b099da0,Site,37.0,...,9fae3479-3cd1-45e1-900f-ede2bdae197f,Michele Cassia De Oliveira,email@email.com,55.0,11,911112222.0,allowed,active,Natalia r.,2023-10-08 11:59:07
13081,NaN,18178,55.618.583/0001-00,Fanny,1409.77,2024-11-29 00:20:04+00:00,Ecommerce,6744bb7c825faf882709f6cc,Link,12.0,...,8e72a9fd-7fdc-45b2-a088-51f64dd4aa7b,Debora Ferreira Da Silveira,email@email.com,55.0,35,911112222.0,allowed,active,Fanny,2024-08-05 12:32:55
13082,NaN,18179,47.483.469/0001-92,Gabi,1481.14,2024-11-29 01:59:01+00:00,Ecommerce,67450e9beb8555d3930e0cfa,Link,13.0,...,088f537f-ab57-41b2-a756-c989e3269246,Michelle Martins Coube Coneglian,email@email.com,55.0,14,911112222.0,allowed,active,Gabi,2024-08-26 15:27:32
